# NB1 — Feature extraction (DINOv2 embeddings)

Extracts 42 windows/study → DINOv2-small patch-token-mean embeddings (fp16), one .npz per study.

**Settings:** Accelerator = GPU T4 x2 (or P100), Internet = OFF.
**Attach:** competition data `rsna-knee-abnormality-detection` + dataset `dinov2-timm-cache`.
Output ~150 MB → save version → publish as dataset `rsna-knee-embeddings-v1`.

Source of truth for windowing: `rsna-knee/src/windows.py` + `src/dataset.py` (round-robin series sampling, 5-slice windows, 224px, 2–98 percentile norm).

In [ ]:
import os, glob
hub_dirs = (glob.glob('/kaggle/input/*/hf/hub') + glob.glob('/kaggle/input/*/*/hf/hub') + glob.glob('/kaggle/input/*/*/*/hf/hub'))
hub_dirs = [h for h in hub_dirs if glob.glob(h + '/models--timm--*')]
assert hub_dirs, f'timm cache not found under /kaggle/input: {sorted(os.listdir("/kaggle/input"))}'
hf_root = os.path.dirname(hub_dirs[0])
os.environ['HF_HOME'] = hf_root
os.environ['TORCH_HOME'] = os.path.join(os.path.dirname(hf_root), 'torch')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('HF_HOME =', hf_root)
print('hub contents:', os.listdir(hub_dirs[0]))


In [ ]:
import os, sys, json, time, glob
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import timm

cands = (glob.glob('/kaggle/input/*/*/train_series.csv') + glob.glob('/kaggle/input/*/train_series.csv') + glob.glob('/kaggle/input/*/*/*/train_series.csv'))
assert cands, f'competition data not attached; mounts={sorted(os.listdir("/kaggle/input"))}'
COMP = Path(os.path.dirname(cands[0]))
print('COMP =', COMP)
OUT_DIR = Path('/kaggle/working/embeddings')
OUT_DIR.mkdir(exist_ok=True)

WINDOWS_PER_STUDY = 42
SLICES_PER_WINDOW = 5
TARGET = 224
DEVICE = 'cuda'
BATCH_STUDIES = 8

df_series = pd.read_csv(COMP / 'train_series.csv')
df_train = pd.read_csv(COMP / 'train.csv')
STUDIES = df_train['StudyInstanceUID'].astype(str).tolist()
series_by_study = {u: g['SeriesInstanceUID'].tolist() for u, g in df_series.groupby('StudyInstanceUID')}
print(f'{len(STUDIES)} studies, {len(df_series)} series')


In [ ]:
# ---- windowing (ported verbatim from src/windows.py) ----
def normalize_percentile(arr, low=2.0, high=98.0):
    p_low = np.percentile(arr, low)
    p_high = np.percentile(arr, high)
    if p_high - p_low < 1e-6:
        return np.zeros_like(arr, dtype=np.float32)
    clipped = np.clip(arr, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low)).astype(np.float32)

def resize_volume(vol, target_size=224):
    d, h, w = vol.shape
    if h == target_size and w == target_size:
        return vol.astype(np.float32)
    resized = np.zeros((d, target_size, target_size), dtype=np.float32)
    for i in range(d):
        resized[i] = cv2.resize(vol[i], (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    return resized

def extract_series_windows(slices, n_windows, slices_per_window=5, target_size=224):
    num_slices = slices.shape[0]
    norm_slices = normalize_percentile(slices)
    resized_slices = resize_volume(norm_slices, target_size=target_size)
    if num_slices < slices_per_window:
        pad = np.repeat(resized_slices[-1:], slices_per_window - num_slices, axis=0)
        resized_slices = np.concatenate([resized_slices, pad], axis=0)
        num_slices = slices_per_window
    max_start = num_slices - slices_per_window
    if max_start == 0:
        start_indices = [0] * n_windows
    else:
        start_indices = np.linspace(0, max_start, n_windows, dtype=int).tolist()
    windows = np.zeros((n_windows, slices_per_window, target_size, target_size), dtype=np.float32)
    for idx, start in enumerate(start_indices):
        windows[idx] = resized_slices[start:start + slices_per_window]
    return windows

def sort_key(p):
    ds = pydicom.dcmread(str(p), stop_before_pixels=True)
    pos = getattr(ds, 'SliceLocation', None)
    if pos is not None:
        return (0, float(pos), 0.0)
    ipp = getattr(ds, 'ImagePositionPatient', None)
    if ipp is not None:
        return (1, 0.0, float(ipp[2]))
    return (2, 0.0, 0.0)

def study_windows(study_uid, series_uids, n_total):
    """Round-robin window budget across series, concatenating per-series windows."""
    num_series = len(series_uids)
    counts = [0] * num_series
    for i in range(n_total):
        counts[i % num_series] += 1
    parts = []
    for series_uid, k in zip(series_uids, counts):
        sdir = COMP / 'train_series' / str(study_uid) / str(series_uid)
        slices_dir = sorted(sdir.glob('*.dcm'), key=sort_key)
        if not slices_dir:
            parts.append(np.zeros((k, SLICES_PER_WINDOW, TARGET, TARGET), dtype=np.float32))
            continue
        vol = np.stack([pydicom.dcmread(str(p)).pixel_array.astype(np.float32) for p in slices_dir])
        parts.append(extract_series_windows(vol, n_windows=k, slices_per_window=SLICES_PER_WINDOW, target_size=TARGET))
    return np.concatenate(parts, axis=0)

In [ ]:
# ---- backbone: timm DINOv2-small, 5-channel input (same as local pipeline) ----
model = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, in_chans=5, img_size=224, num_classes=0)
model = model.to(DEVICE).eval()
PREFIX = int(getattr(model, 'num_prefix_tokens', 1))
print('prefix tokens:', PREFIX)

@torch.no_grad()
def embed_windows(win_np):
    """win_np: [W,5,224,224] -> [W,384] fp16 numpy"""
    out = []
    for i in range(0, len(win_np), 64):
        x = torch.from_numpy(win_np[i:i+64]).to(DEVICE)
        feats = model.forward_features(x)
        emb = feats[:, PREFIX:].mean(dim=1)
        out.append(emb.cpu().numpy().astype(np.float16))
    return np.concatenate(out, axis=0)

In [ ]:
# ---- main extraction loop ----
t0 = time.time()
done = 0
for study_uid in STUDIES:
    dest = OUT_DIR / f'{study_uid}.npz'
    if dest.exists():
        done += 1
        continue
    series_uids = series_by_study.get(study_uid, [])
    if series_uids:
        win = study_windows(study_uid, series_uids, WINDOWS_PER_STUDY)
    else:
        win = np.zeros((WINDOWS_PER_STUDY, SLICES_PER_WINDOW, TARGET, TARGET), dtype=np.float32)
    emb = embed_windows(win)
    np.savez_compressed(dest, emb=emb)
    done += 1
    if done % 100 == 0:
        el = time.time() - t0
        print(f'{done}/{len(STUDIES)}  {el/60:.1f} min  eta {el/done*(len(STUDIES)-done)/60:.0f} min', flush=True)
print('DONE', done, 'in', (time.time()-t0)/60, 'min')

In [ ]:
# ---- manifest ----
rows = []
for p in sorted(OUT_DIR.glob('*.npz')):
    e = np.load(p)['emb']
    rows.append({'StudyInstanceUID': p.stem, 'n_windows': int(e.shape[0]), 'dim': int(e.shape[1])})
pd.DataFrame(rows).to_csv('/kaggle/working/manifest.csv', index=False)
print(len(rows), 'studies embedded')